In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import common
import sconv

from data import get_loader_cifar10, get_loader_imagenet
from resnet import ResNet
from resnet_imagenet import ResNet_ImageNet

os.environ["CUDA_VISIBLE_DEVICES"] = '2'

# need to perfect
def make_model(args):
    conv1x1 = common.default_conv
    conv3x3 = sconv.scale_conv
    if args.dataset == 'cifar10':
        BaseNet = ResNet
    elif args.dataset == 'imagenet':
        BaseNet = ResNet_ImageNet
    
    class ScaleNet(BaseNet):
        def __init__(self, args):
            super(ScaleNet, self).__init__(args=args, conv3x3=conv3x3, conv1x1=conv1x1)
            
            if args.dataset == 'cifar10':
                parent = ResNet(args)
            elif args.dataset == 'imagenet':
                parent = ResNet_ImageNet(args)
                
            _, s_dict = self.extract(parent)
            # load the sign of model
            self.quantize_kernels(parent, Bnn_Dict, s_dict,
                                  args.input_bits, args.scale_bits)

        def gen(self, target, conv1x1=False):
            def _criterion(m):
                if isinstance(m, nn.Conv2d):
                    if conv1x1:
                        return m.kernel_size[0] * m.kernel_size[1] == 1
                    else:
                        return m.kernel_size[0] * m.kernel_size[1] > 1
                elif isinstance(m, nn.ConvTranspose2d):
                    return True
                return False
            
            gen = (m for m in target.modules() if _criterion(m))

            return gen

        def extract(self, parent):
            k_dict = {}
            s_dict = {}
            modules = [m for m in self.gen(parent)]
            
            for k, m in enumerate(modules):
                weights, scales = self.preprocess_kernels(m)
                k_dict[k] = weights
                s_dict[k] = nn.Parameter(scales, requires_grad=args.is_train)

            return k_dict, s_dict
    
        def preprocess_kernels(self, m):
            c_out, c_in, kh, kw = m.weight.size()
            weights = m.weight.data.view(c_out, c_in, kh * kw)
            scales = weights.norm(2, dim=2)
            scales.unsqueeze_(-1)
            scales.unsqueeze_(-1)  
            weights = weights.view(c_out * c_in, kh * kw)

            return weights, scales

        def quantize_kernels(
            self, parent, k_dict, s_dict, input_bits, scale_bits):
            modules_parent = [m for m in self.gen(parent)]
            modules_self = [m for m in sconv.gen_sconv(self)]
            
            for k, v in enumerate(modules_self):
                source = modules_parent[k]
                target = modules_self[k]

                target.set_params(
                    source,
                    kernels=k_dict[k],
                    scales=s_dict[k],
                    input_bits=input_bits,
                    scale_bits=scale_bits,
                )

        def state_dict(self, destination=None, prefix='', keep_vars=False):
            state = super(ScaleNet, self).state_dict(
                destination=destination, prefix=prefix, keep_vars=keep_vars)

            return state

        def load_state_dict(self, state_dict, strict=True, init=False):
            super(ScaleNet, self).load_state_dict(state_dict, strict=False)

    return ScaleNet(args)


# CIFAR-10

In [2]:
# declare the args
class A(object):
    def __init__(self):
        self.n_threads = 10
        self.dataset = 'cifar10'
        self.input_bits = 32
        self.activation = 'relu'
        self.model = 'resnet'
        self.depth = 18
        self.scale_bits = 16
        self.is_train = False
        self.dir_data = './Dataset'

a = A()

In [3]:
# load test data
loader_train, loader_test = get_loader_cifar10(a)

Files already downloaded and verified


In [4]:
# load the sign and the scaling factor
model_dir = './Result/cifar10/'
device = torch.device('cuda')
Bnn_Dict = torch.load(model_dir + 'bnn_dict.pth')
weight_dict = torch.load(model_dir + 'model_best.pth',map_location=device)

model = make_model(a).to(device)
model.load_state_dict(weight_dict)

In [5]:
print("Waiting Test!")
with torch.no_grad():
    correct = 0
    total = 0
    for data in loader_test:
        model.eval()
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum()
    print('Accuracy：%.3f%%' % (100 * correct.float() / total))

Waiting Test!
Accuracy：94.030%


# ImageNet

In [2]:
# declare the args
class A(object):
    def __init__(self):
        self.n_threads = 10
        self.dataset = 'imagenet'
        self.input_bits = 32
        self.activation = 'relu'
        self.model = 'resnet'
        self.depth = 18
        self.scale_bits = 16
        self.is_train = False
        self.dir_data = '/root/datas/'

a = A()

In [3]:
# load test data
loader_train, loader_test = get_loader_imagenet(a.dir_data)

In [4]:
# load the sign and the scaling factor
model_dir = './Result/imagenet/'
device = torch.device('cuda')
Bnn_Dict = torch.load(model_dir + 'bnn_dict.pth')
weight_dict = torch.load(model_dir + 'model_best.pth',map_location=device)

model = make_model(a).to(device)
model.load_state_dict(weight_dict)

In [5]:
print("Waiting Test!")
with torch.no_grad():
    correct = 0
    total = 0
    for data in loader_test:
        model.eval()
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum()
    print('Accuracy：%.3f%%' % (100 * correct.float() / total))

Waiting Test!
Accuracy：67.012%
